## Web-scraping : books.toscrape.com ##

1. Pas d'API pi d'open data
2. Pas de tobots.txt : tout est permis
3. Pas de GCU (site fait pour)
4. Pas de données personnelles
5. 1 requête par seconde maximum
6. Je vais enregistrer la collecte en CSV


48 livres, 3 pages

Sélecteur : article.product_pod -> 20

Donnée			    Sélecteur		    Contenu ou attribut ?	    nb ds console

Titre			    h3 a			    attribut : title		    20
Prix			    p.price_color		contenu				        20
Disponibilité		p.instock		    contenu				        20
Note			    p.star-rating		classe				        20
Lien vers la fiche	h3 a			    attribut : href			    20

Page "Next"		    li a			    attribut : href="page-2.html"


Je trouve bien le prix dans le code source, ce qui signifie que je peux le récupérer avec un script.

Le stock et la description sont deux informations disponibles sur la fiche détaillée mais pas sur la liste précédente.

Stock : Sélecteur = p.instock (contenu)
Description : Sélecteur = p (contenu)

In [27]:
#Cours
import requests

URL = "https://books.toscrape.com/catalogue/category/books/fantasy_19/index.html"
ENTETES = {"User-Agent": "bootcamp-data/1.0 (jordanpiade@gmail.com)"}

r = requests.get(URL, headers = ENTETES, timeout = 10)
r.encoding = 'utf-8'

print("status :", r.status_code)
print("longueur :", len(r.text), "caracteres")
print("debut :", r.text[:60])

status : 200
longueur : 51222 caracteres
debut : 

<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" 


In [28]:
print("Le prix est présent :", "18.78" in r.text)
print("le titre est présent :", "Unicorn Tracks" in r.text)

Le prix est présent : True
le titre est présent : True


In [29]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(r.text, "html.parser")

fiches = soup.select("article.product_pod")
print("fiches trouvees :", len(fiches))

fiches trouvees : 20


In [30]:
fiche = fiches[0]

prix = fiche.select_one("p.price_color").get_text(strip = True)

lien_h3 = fiche.select_one("h3 a")
titre = lien_h3["title"]
href = lien_h3["href"]

print("prix :", prix)
print("titre :", titre)
print("href :", href)

prix : £18.78
titre : Unicorn Tracks
href : ../../../unicorn-tracks_951/index.html


In [31]:
classes = fiche.select_one("p.star-rating")["class"]
print("classes :", classes)

classes : ['star-rating', 'Three']


In [32]:
NOTES = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

note = NOTES[classes[1]]
print("note :", note)

note : 3


In [33]:
from urllib.parse import urljoin

print(urljoin(URL, href))

https://books.toscrape.com/catalogue/unicorn-tracks_951/index.html


In [34]:
def lire_fiche(fiche, url_page):
    lien = fiche.select_one("h3 a")
    classes = fiche.select_one("p.star-rating")["class"]
    return {
        "titre": lien["title"],
        "prix": fiche.select_one("p.price_color").get_text(strip=True),
        "dispo": fiche.select_one("p.instock").get_text(strip=True),
        "note": NOTES[classes[1]],
        "url": urljoin(url_page, lien["href"]),
    }


print(lire_fiche(fiches[0], URL))

{'titre': 'Unicorn Tracks', 'prix': '£18.78', 'dispo': 'In stock', 'note': 3, 'url': 'https://books.toscrape.com/catalogue/unicorn-tracks_951/index.html'}


In [35]:
suivant = soup.select_one("li.next a")
print(suivant["href"] if suivant else "pas de page suivante")

page-2.html


In [ ]:
import time

def collecter_toutes_les_pages(url_depart, pages_max=3):
    url = url_depart
    livres = []
    page = 0

    while url and page < pages_max:
        page += 1
        r = requests.get(url, headers=ENTETES, timeout=10)
        r.encoding = "utf-8"
        if r.status_code != 200:
            print(f"  page {page} : status {r.status_code}, on s'arrete")
            break

        soup = BeautifulSoup(r.text, "html.parser")
        fiches = soup.select("article.product_pod")
        livres += [lire_fiche(f, url) for f in fiches]
        print(f"  page {page} : {len(fiches)} livres  (total {len(livres)})")

        suivant = soup.select_one("li.next a")
        url = urljoin(url, suivant["href"]) if suivant else None

        time.sleep(1)

    return livres


livres = collecter_toutes_les_pages("https://books.toscrape.com/catalogue/page-1.html")
print("collecte :", len(livres), "livres")

  page 1 : 20 livres  (total 20)
  page 2 : 20 livres  (total 40)
  page 3 : 20 livres  (total 60)
collecte : 60 livres


In [37]:
def lire_detail(url):
    r = requests.get(url, headers=ENTETES, timeout=10)
    r.encoding = "utf-8"
    if r.status_code != 200:
        return {"categorie": None, "stock": None}

    soup = BeautifulSoup(r.text, "html.parser")

    # le tableau : une entree par ligne, le th en cle et le td en valeur
    infos = {}
    for ligne in soup.select("table.table-striped tr"):
        cle = ligne.select_one("th")
        val = ligne.select_one("td")
        if cle and val:
            infos[cle.get_text(strip=True)] = val.get_text(strip=True)

    # la categorie : troisieme element du fil d'Ariane
    miettes = soup.select("ul.breadcrumb li a")
    categorie = miettes[-1].get_text(strip=True) if miettes else None

    return {
        "categorie": categorie,
        "stock": infos.get("Availability"),
        "upc": infos.get("UPC"),
    }


url_test = "https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html"
print(lire_detail(url_test))

{'categorie': 'Poetry', 'stock': 'In stock (22 available)', 'upc': 'a897fe39b1053632'}


In [38]:
import re

def nombre_en_stock(texte):
    if not texte:
        return None
    trouve = re.search(r"\((\d+) available\)", texte)
    return int(trouve.group(1)) if trouve else 0


print(nombre_en_stock("In stock (22 available)"))
print(nombre_en_stock("Out of stock"))
print(nombre_en_stock(None))

22
0
None


Etape 2 :

In [39]:
import pandas as pd

page_une = pd.DataFrame(lire_fiche(fiche, URL) for fiche in fiches)
page_une.head()


,titre,prix,dispo,note,url
0,Unicorn Tracks,£18.78,In stock,3,https://books.toscrape.com/catalogue/unicorn-t...
1,"Saga, Volume 6 (Saga (Collected Editions) #6)",£25.02,In stock,3,https://books.toscrape.com/catalogue/saga-volu...
2,Princess Between Worlds (Wide-Awake Princess #5),£13.34,In stock,5,https://books.toscrape.com/catalogue/princess-...
3,Masks and Shadows,£56.40,In stock,2,https://books.toscrape.com/catalogue/masks-and...
4,Crown of Midnight (Throne of Glass #2),£43.29,In stock,3,https://books.toscrape.com/catalogue/crown-of-...


Etape 3 :

In [42]:
livres = collecter_toutes_les_pages(URL)
print("collecte :", len(livres), "livres")

  page 1 : 20 livres  (total 20)
  page 2 : 20 livres  (total 40)
  page 3 : 8 livres  (total 48)
collecte : 48 livres


In [43]:
fantasy_livres = pd.DataFrame(livres)
fantasy_livres.head()

,titre,prix,dispo,note,url
0,Unicorn Tracks,£18.78,In stock,3,https://books.toscrape.com/catalogue/unicorn-t...
1,"Saga, Volume 6 (Saga (Collected Editions) #6)",£25.02,In stock,3,https://books.toscrape.com/catalogue/saga-volu...
2,Princess Between Worlds (Wide-Awake Princess #5),£13.34,In stock,5,https://books.toscrape.com/catalogue/princess-...
3,Masks and Shadows,£56.40,In stock,2,https://books.toscrape.com/catalogue/masks-and...
4,Crown of Midnight (Throne of Glass #2),£43.29,In stock,3,https://books.toscrape.com/catalogue/crown-of-...


Etape 4 :

In [ ]:
details = (fantasy_livres['url'].apply(lire_detail))
details.head()

0    {'categorie': 'Fantasy', 'stock': 'In stock (1...
1    {'categorie': 'Fantasy', 'stock': 'In stock (1...
2    {'categorie': 'Fantasy', 'stock': 'In stock (1...
3    {'categorie': 'Fantasy', 'stock': 'In stock (1...
4    {'categorie': 'Fantasy', 'stock': 'In stock (1...
Name: url, dtype: object

In [48]:
details = pd.DataFrame(dico for dico in details)
details.head()

,categorie,stock,upc
0,Fantasy,In stock (16 available),7ae099f3898e0209
1,Fantasy,In stock (16 available),c849f0b2f5d6a742
2,Fantasy,In stock (16 available),0e691eda369f4e09
3,Fantasy,In stock (16 available),fa9610a50a1bf149
4,Fantasy,In stock (16 available),7267864328ae8b9d


In [49]:
details['stock'] = details['stock'].apply(nombre_en_stock)
details.head()

,categorie,stock,upc
0,Fantasy,16,7ae099f3898e0209
1,Fantasy,16,c849f0b2f5d6a742
2,Fantasy,16,0e691eda369f4e09
3,Fantasy,16,fa9610a50a1bf149
4,Fantasy,16,7267864328ae8b9d


Etape 5 :

In [ ]:
fantasy_livres = pd.concat([fantasy_livres, details], axis = 1)
fantasy_livres.head()

,titre,prix,dispo,note,url,categorie,stock,upc
0,Unicorn Tracks,£18.78,In stock,3,https://books.toscrape.com/catalogue/unicorn-t...,Fantasy,16,7ae099f3898e0209
1,"Saga, Volume 6 (Saga (Collected Editions) #6)",£25.02,In stock,3,https://books.toscrape.com/catalogue/saga-volu...,Fantasy,16,c849f0b2f5d6a742
2,Princess Between Worlds (Wide-Awake Princess #5),£13.34,In stock,5,https://books.toscrape.com/catalogue/princess-...,Fantasy,16,0e691eda369f4e09
3,Masks and Shadows,£56.40,In stock,2,https://books.toscrape.com/catalogue/masks-and...,Fantasy,16,fa9610a50a1bf149
4,Crown of Midnight (Throne of Glass #2),£43.29,In stock,3,https://books.toscrape.com/catalogue/crown-of-...,Fantasy,16,7267864328ae8b9d


In [53]:
fantasy_livres['prix'] = fantasy_livres['prix'].apply(lambda x: float(x[1:]))
fantasy_livres.dtypes

titre            str
prix         float64
dispo            str
note           int64
url              str
categorie        str
stock          int64
upc              str
dtype: object

Etape 6 :

In [54]:
print(f"{len(fantasy_livres)} livres au total.")
print(f"Stock total : {sum(fantasy_livres['stock'])}.")

48 livres au total.
Stock total : 372.


In [67]:
import statistics
print(f"Prix moyen : {statistics.mean(fantasy_livres['prix']):.2f} €.")
print(f"Livre le plus cher : {fantasy_livres[fantasy_livres['prix'] == fantasy_livres['prix'].max()]['titre'].iloc[0]}.")

Prix moyen : 39.59 €.
Livre le plus cher : Myriad (Prentor #1).
